# HSeeker component ablation and parameter variation (reviewer comment R3.9)

Runs on the balanced benchmark `hdna_benchmark_balanced_v3.csv` (130 records, 65 forming / 65
non-forming; 70 experimental + 60 designed negatives) at the shipped defaults: minimum arm 8,
maximum spacer 10, purity 0.90, mismatch 0.10, AT threshold 0.80, homopolymer filter on.

A record is called forming when at least one candidate survives; no score threshold is used. AUROC
values are exploratory, within-cohort descriptions, not independent estimates of predictive accuracy.
The same functions are in `analysis/scripts/run_ablation_comment9.py`, which writes the tables to
`analysis/results/ablation_comment9/`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "hdna_benchmark_balanced_v3.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "analysis" / "scripts"))
import run_ablation_comment9 as ab
records = ab.load(ROOT / "hdna_benchmark_balanced_v3.csv")
print(len(records), "records;", sum(r["label"] == "forming" for r in records), "forming")

130 records; 65 forming


## 1. Component ablation (cumulative stages, overlaps retained)

In [2]:
calls, stages = ab.run_stages(records)
cols = ["stage", "subset", "TP", "FN", "TN", "FP", "sensitivity", "sensitivity_ci95",
        "specificity", "specificity_ci95", "MCC", "AUROC_exploratory"]
pd.DataFrame(stages)[cols]

,stage,subset,TP,FN,TN,FP,sensitivity,sensitivity_ci95,specificity,specificity_ci95,MCC,AUROC_exploratory
0,detection,balanced,63,2,58,7,0.9692,"[0.8946, 0.9915]",0.8923,"[0.794, 0.9468]",0.8641,NaN
1,detection,experimental,63,2,4,1,0.9692,"[0.8946, 0.9915]",0.8000,"[0.3755, 0.9638]",0.7077,NaN
2,+purity,balanced,63,2,65,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.9442, 1.0]",0.9697,NaN
3,+purity,experimental,63,2,5,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.5655, 1.0]",0.8321,NaN
4,+AT_filter,balanced,63,2,65,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.9442, 1.0]",0.9697,NaN
5,+AT_filter,experimental,63,2,5,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.5655, 1.0]",0.8321,NaN
6,pairing,balanced,63,2,65,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.9442, 1.0]",0.9697,0.9846
7,pairing,experimental,63,2,5,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.5655, 1.0]",0.8321,0.9846
8,pairing_stacking,balanced,63,2,65,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.9442, 1.0]",0.9697,0.9846
9,pairing_stacking,experimental,63,2,5,0,0.9692,"[0.8946, 0.9915]",1.0000,"[0.5655, 1.0]",0.8321,0.9846


Non-forming records called by the detector alone (removed by the purity requirement):

In [3]:
pd.DataFrame([c for c in calls if c["stage"] == "detection" and c["label"] == "non_forming" and c["call"]])[
    ["record_id", "sequence_name", "record_type", "candidates"]]

,record_id,sequence_name,record_type,candidates
0,HDNA0028,B-33,experimental,3
1,SYN0042,G4_G3-tracts_TT-loops,synthetic,1
2,SYN0046,Z-DNA_(CG)10,synthetic,4
3,SYN0047,Z-DNA_(CG)12,synthetic,8
4,SYN0057,perfect_mirror_balancedGC_1,synthetic,2
5,SYN0058,perfect_mirror_balancedGC_2,synthetic,2
6,SYN0059,perfect_mirror_balancedGC_3,synthetic,2


## 2. Overlap handling at the shipped configuration

In [4]:
pd.DataFrame(ab.run_overlap_modes(records))[["overlap_mode", "subset", "TP", "FN", "TN", "FP", "MCC", "AUROC_exploratory"]]

,overlap_mode,subset,TP,FN,TN,FP,MCC,AUROC_exploratory
0,retain_all,balanced,63,2,65,0,0.9697,0.9846
1,retain_all,experimental,63,2,5,0,0.8321,0.9846
2,greedy,balanced,63,2,65,0,0.9697,0.9846
3,greedy,experimental,63,2,5,0,0.8321,0.9846
4,score,balanced,63,2,65,0,0.9697,0.9846
5,score,experimental,63,2,5,0,0.8321,0.9846


## 3. One parameter at a time (others at the defaults)

In [5]:
vary = pd.DataFrame(ab.run_vary(records))
vary[vary.subset == "balanced"][["parameter", "value", "is_default", "TP", "FN", "TN", "FP", "MCC"]]

,parameter,value,is_default,TP,FN,TN,FP,MCC
0,purity,0.80,False,63,2,65,0,0.9697
2,purity,0.85,False,63,2,65,0,0.9697
4,purity,0.90,True,63,2,65,0,0.9697
6,purity,0.95,False,54,11,65,0,0.8429
8,purity,1.00,False,54,11,65,0,0.8429
10,mismatch,0.00,False,49,16,65,0,0.7778
12,mismatch,0.05,False,49,16,65,0,0.7778
14,mismatch,0.10,True,63,2,65,0,0.9697
16,mismatch,0.15,False,64,1,64,1,0.9692
18,mismatch,0.20,False,64,1,64,1,0.9692


## 4. Joint grid and out-of-sample selection

The 576-configuration grid with selection inside training folds is produced by
`analysis/scripts/run_parameter_robustness.py` and `run_grouped_validation.py`.

In [6]:
pd.read_csv(ROOT / "analysis/results/robustness_v3/nested_cv_summary.csv")[
    ["config_name", "MCC_median", "TP", "FN", "TN", "FP", "MCC_boot_lo", "MCC_boot_hi"]]

,config_name,MCC_median,TP,FN,TN,FP,MCC_boot_lo,MCC_boot_hi
0,NESTED-CV (per-fold tuned),0.9389,62,3,64,1,0.8766,0.9847
1,fixed: script_default,0.9697,63,2,65,0,0.9254,1.0000
2,fixed: production_cli,0.8836,57,8,65,0,0.8054,0.9544
3,fixed: manuscript_stated,0.9692,64,1,64,1,0.9230,1.0000


In [7]:
pd.read_csv(ROOT / "analysis/results/grouped_validation_v3/grouped_validation_metrics.csv")[
    ["level", "scheme", "n", "TP", "FN", "TN", "FP", "MCC"]]

,level,scheme,n,TP,FN,TN,FP,MCC
0,family,family: per-fold tuned,130,61,4,64,1,0.9241
1,family,family: fixed script_default,130,63,2,65,0,0.9697
2,family,family: fixed production_cli,130,57,8,65,0,0.8836
3,family,family: fixed manuscript_stated,130,64,1,64,1,0.9692
4,study,study: per-fold tuned,130,61,4,64,1,0.9241
5,study,study: fixed script_default,130,63,2,65,0,0.9697
6,study,study: fixed production_cli,130,57,8,65,0,0.8836
7,study,study: fixed manuscript_stated,130,64,1,64,1,0.9692
8,collapsed,collapsed: fixed script_default,52,43,2,7,0,0.8621
9,collapsed,collapsed: fixed production_cli,52,38,7,7,0,0.6498
